<a href="https://colab.research.google.com/github/hannay-sen/NEUROSCAN/blob/main/notebooks/colab_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroScan Agent: end-to-end training on Colab
Runtime → Change runtime type → **GPU**. Everything is saved to Google Drive so it survives disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/neuroscan'
!mkdir -p $DRIVE/data/raw/kaggle $DRIVE/data/raw/figshare $DRIVE/checkpoints $DRIVE/results

Mounted at /content/drive


In [ ]:
# Replace with your repo URL
!git clone https://github.com/hannay-sen/NEUROSCAN.git /content/neuroscan
%cd /content/neuroscan
!pip install -q -r requirements.txt && pip install -q -e .

Cloning into '/content/neuroscan'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 68 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 48.74 KiB | 2.44 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/neuroscan
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to b

## Link data, checkpoints and results to Drive
Put the Kaggle *Brain Tumor MRI Dataset* folders (`Training/`, `Testing/`) in `MyDrive/neuroscan/data/raw/kaggle/` and the Figshare `.mat` files in `MyDrive/neuroscan/data/raw/figshare/`.

In [ ]:
!rm -rf data checkpoints results
!ln -s $DRIVE/data data
!ln -s $DRIVE/checkpoints checkpoints
!ln -s $DRIVE/results results
!ls data/raw/kaggle data/raw/figshare | head

data/raw/figshare:
1000.mat
1001.mat
1002.mat
1003.mat
1004.mat
1005.mat
1006.mat
1007.mat
1008.mat


## (Optional) download Kaggle data with the Kaggle API
Upload `kaggle.json` from your Kaggle account settings first.

In [ ]:
# from google.colab import files; files.upload()   # kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p data/raw/kaggle --unzip

## 1. Data

In [5]:
!python -m neuroscan.data.prepare_figshare
!python -m neuroscan.data.dedup
!python -m neuroscan.data.make_splits

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/neuroscan/src/neuroscan/data/prepare_figshare.py", line 74, in <module>
    main()
    ~~~~^^
  File "/content/neuroscan/src/neuroscan/data/prepare_figshare.py", line 63, in main
    Image.fromarray((d["mask"] > 0).astype(np.uint8) * 255).save(msk_p)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/PIL/Image.py", line 2583, in save
    fp = builtins.open(filename, "w+b")
KeyboardInterrupt
^C
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/neuroscan/src/neuroscan/data/dedup.py", line 112, in <module>
    main()
    ~~~~^^
  File "/content/neuroscan/src/neuroscan/data/dedup.py", line 61, in main
    kg = scan_kaggle(path("kaggle_raw"), cfg["classes"])
  Fil

In [6]:
import pandas as pd
pd.read_csv('data/splits/dedup_pairs.csv').sample(10, random_state=0)   # spot-check matches

,kind,a,b,hamming
1830,kaggle_internal,/content/neuroscan/data/raw/kaggle/Training/no...,/content/neuroscan/data/raw/kaggle/Training/no...,10
361,kaggle_vs_figshare,/content/neuroscan/data/raw/kaggle/Training/me...,/content/neuroscan/data/processed/figshare/ima...,9
1191,kaggle_vs_figshare,/content/neuroscan/data/raw/kaggle/Training/pi...,/content/neuroscan/data/processed/figshare/ima...,4
2244,kaggle_internal,/content/neuroscan/data/raw/kaggle/Training/no...,/content/neuroscan/data/raw/kaggle/Training/no...,12
2533,kaggle_internal,/content/neuroscan/data/raw/kaggle/Testing/gli...,/content/neuroscan/data/raw/kaggle/Testing/gli...,0
2722,kaggle_internal,/content/neuroscan/data/raw/kaggle/Testing/not...,/content/neuroscan/data/raw/kaggle/Training/no...,1
483,kaggle_vs_figshare,/content/neuroscan/data/raw/kaggle/Training/me...,/content/neuroscan/data/processed/figshare/ima...,5
306,kaggle_vs_figshare,/content/neuroscan/data/raw/kaggle/Training/me...,/content/neuroscan/data/processed/figshare/ima...,6
1363,kaggle_vs_figshare,/content/neuroscan/data/raw/kaggle/Training/pi...,/content/neuroscan/data/processed/figshare/ima...,4
118,kaggle_vs_figshare,/content/neuroscan/data/raw/kaggle/Training/gl...,/content/neuroscan/data/processed/figshare/ima...,8


In [7]:
%cd /content/neuroscan
!rm data && cp -r $DRIVE/data data
!du -sh data

/content/neuroscan
2.2G	data


## 2. Train

In [8]:
!python -m neuroscan.train_classifier
!python -m neuroscan.train_classifier --balance none --tag nobal

I0000 00:00:1795630850.203808   12432 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795630853.779800   12432 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

model.safetensors: downloading bytes:  36% 7.72M/21.4M [00:00<00:01, 12.8MB/s,  259kB/s  ]
model.safetensors: reconstructing file:  23% 4.89M/21.4M [00:00<00:02, 5.88MB/s]
model.safetensors: downloading bytes:  77% 16.4M/21.4M [00:00<00:00, 23.4MB/s,  987kB/s  ]
model.safetensors: reconstructing file:  48% 10.3M/21.4M [00:01<00:01, 8.91MB/s,  845kB/s  ]
model.safetensors: downloading bytes: 100% 20.2M/20.2M [00:01<00:00, 15.1MB/s, 1.89MB/s  ]
model.safetensors: reconstructing file: 100% 21.4M/21.4M [00:01<00:00, 16.0MB/s, 2.03MB/s  ]
epoch 01  loss 0.8106  val_acc 0

In [9]:
!python -m neuroscan.train_segmentation

I0000 00:00:1795631736.846932   16754 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795631739.577230   16754 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/neuroscan/src/neuroscan/train_segmentation.py", line 84, in <module>
    main()
    ~~~~^^
  File "/content/neuroscan/src/neuroscan/train_segmentation.py", line 58, in main
    for x, y in train_dl:
                ^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 725, in __next__
    data = self._next_data()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/

In [10]:
!cp -r checkpoints $DRIVE/

cp: 'checkpoints' and '/content/drive/MyDrive/neuroscan/checkpoints' are the same file


In [11]:
!rm -rf data/processed/figshare

In [12]:
!rm -rf data/processed/figshare

In [13]:
!python -m neuroscan.data.prepare_figshare

skip cvind.mat: "Unable to synchronously open object (object 'cjdata' doesn't exist)"
Wrote 3064 slices from 233 patients -> /content/neuroscan/data/processed/figshare/figshare.csv
label
glioma        1426
pituitary      930
meningioma     708


In [14]:
!python -m neuroscan.data.dedup

Hashing 7200 Kaggle images...
Hashing 3064 Figshare images...
Removed 3005 of 7200 Kaggle images (1731 Figshare-overlap pairs, 1277 internal-duplicate pairs).
Kept 4195 -> /content/neuroscan/data/splits/kaggle_clean.csv


In [15]:
!python -m neuroscan.data.make_splits

kaggle_train: 2781  {'glioma': 1001, 'meningioma': 667, 'notumor': 570, 'pituitary': 543}
kaggle_val: 491  {'glioma': 177, 'meningioma': 118, 'notumor': 100, 'pituitary': 96}
kaggle_test: 923  {'pituitary': 322, 'glioma': 288, 'meningioma': 272, 'notumor': 41}
figshare_train: 2144 slices, 163 patients
figshare_val: 449 slices, 35 patients
figshare_test: 471 slices, 35 patients


In [16]:
!python -m neuroscan.train_segmentation

I0000 00:00:1795632280.869559   19174 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795632283.626901   19174 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
epoch 01  loss 1.5829  val_dice 0.3421  (33s)
epoch 02  loss 1.4911  val_dice 0.4608  (32s)
epoch 03  loss 1.4289  val_dice 0.5222  (33s)
epoch 04  loss 1.3689  val_dice 0.5417  (33s)
epoch 05  loss 1.3075  val_dice 0.5628  (32s)
epoch 06  loss 1.2432  val_dice 0.5304  (31s)
epoch 07  loss 1.0514  val_dice 0.6067  (32s)
epoch 08  loss 0.7733  val_dice 0.6268  (32s)
epoch 09  loss 0.5537  val_dice 0.6597  (32s)
epoch 10  loss 0.4231  val_dice 0.6823  (32s)
epoch 11  loss 0.3507  val_dice 0.6967  (32s)
epoch 12  loss 0.3133  val_dice 0.7071  (32s)
epoch 13  loss 0.2928

## 3. Evaluate

In [17]:
!python -m neuroscan.eval.evaluate_classifier --split kaggle_test
!python -m neuroscan.eval.evaluate_classifier --split figshare_all
!python -m neuroscan.eval.evaluate_classifier --split kaggle_test --tag nobal
!python -m neuroscan.eval.evaluate_segmentation --split figshare_test
!python -m neuroscan.eval.evaluate_safety --split kaggle_val --sweep

I0000 00:00:1795632807.966802   21713 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795632811.082026   21713 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
accuracy                     0.9166
macro_f1_present_classes     0.8855
ece_uncalibrated             0.0651
ece_calibrated               0.0431
-> results/classifier_kaggle_test.json
-> results/classifier_kaggle_test_confusion.png
I0000 00:00:1795632833.628565   21847 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795632836.523343   21847 c

Copy the chosen `min_confidence` / `max_entropy` into `configs/config.yaml`, then:

In [18]:
!python -m neuroscan.eval.evaluate_safety --split kaggle_test
!ls results

I0000 00:00:1795632977.313218   22557 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795632980.280596   22557 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
min_confidence         0.6
max_entropy            0.5
coverage               0.9317443120260022
flag_rate              0.06825568797399784
error_recall           0.44155844155844154
error_rate_overall     0.08342361863488625
error_rate_unflagged   0.05
error_rate_flagged     0.5396825396825397
split                  kaggle_test
-> results/safety_kaggle_test.json
classifier_figshare_all_confusion.png	    safety_kaggle_test.json
classifier_figshare_all.json		    safety_sweep_kaggle_val.json
classifier_kaggle_test_confusion.png	    segmentation_figshare_test.csv
classif

## 4. Try one scan

In [19]:
import glob
!python -m neuroscan.graph {glob.glob('data/raw/kaggle/Testing/glioma/*')[0]}

I0000 00:00:1795633118.436533   23189 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795633120.906090   23189 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
{
  "image": "data/raw/kaggle/Testing/glioma/Te-gl_117.jpg",
  "disclaimer": "Research prototype for decision support only. Not a medical device and not for clinical diagnosis. Every output must be confirmed by a qualified radiologist.",
  "needs_review": true,
  "flags": [
    "no_knowledge_retrieved"
  ],
  "flag_details": {
    "no_knowledge_retrieved": "no knowledge-base passages were found for this tumor type"
  },
  "classification": {
    "label": "glioma",
    "confidence": 1.0,
    "probs": {
      "glioma": 1.0,
      "meningioma": 0.0,
      "notumor": 0.0

In [20]:
!cp -r results checkpoints configs $DRIVE/

cp: 'results' and '/content/drive/MyDrive/neuroscan/results' are the same file
cp: 'checkpoints' and '/content/drive/MyDrive/neuroscan/checkpoints' are the same file
